# Example as from the CFM training script.

## Preparing notebook.

### Libraries.

In [1]:
import os
import sys
import yaml

from hydra import initialize, compose
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import seaborn as sns
import torch

from sc_exp_design.config import NeuralVelocityFieldConfig
from sc_exp_design.constants import DataFields, PredictionFields
from sc_exp_design.data.container import DataMixin
from sc_exp_design.models import FlowMatching
from sc_exp_design.training.callbacks import WandBLogger, MetricsCallBack, TrainingCallBacks

sys.path.insert(0, "../../../scripts")

from data_utils import annotate_perturbations, annotate_cell_state_data, apply_shared_transformations
from ood_utils import split_adata
from train_utils import (
    parse_mlp_config_dictionary,
    parse_nested_mlp_config_dictionary,
    resolve_omegaconf_to_dictionary
)
from validation_utils import validate_on_ood_data

### Constants.

In [2]:
IS_RUN_LOCALLY = False

time_samplers = {}
noise_distributions = {}
activation_functions = {}
optimizers = {}
schedulers = {}
state_transforms = {}
column2tranform = {}


### Reading configurations.

In [3]:
# Initialize Hydra with your config directory
with initialize(
    config_path="../config"
):
    # Compose the config (like what @hydra.main does)
    config = compose(config_name="train_cfm")

/tmp/ipykernel_3378354/630818722.py:2: UserWarning: 
The version_base parameter is not specified.
Please specify a compatibility version level, or None.
Will assume defaults for version 1.1
  with initialize(


## Data.

### Reading data.

In [4]:
adata = sc.read_h5ad(config.paths.h5ad_path)
adata

AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
    obsm: 'X_pca', 'X_umap'
    layers: 'positives', 'raw'
    

### Annotating perturbations.

In [5]:
adata = annotate_perturbations(
    adata,
    config.annotation.protocol_columns,
    protocol_obs_key_added=config.annotation.protocol_obs_key_added,
    protocol_sep=config.annotation.protocol_sep,
    one_hot_uns_key_added=config.annotation.one_hot_uns_key_added,
    column2tranform=column2tranform,
    protocol_obsm_key=config.annotation.protocol_obsm_key,
)
adata

AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type', 'protocol_id'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap', 'one_hot', '740-yp_[µm]_one_hot', 'mtg_[µm]_one

### Annotating state data.

In [6]:
adata = annotate_cell_state_data(
    adata,
    config.annotation.scatter_columns,
    base_sample_rep=config.annotation.base_sample_rep,
    scatter_obsm_key=config.annotation.scatter_obsm_key,
    concat_obsm_key=config.annotation.concat_obsm_key,
)
adata

AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type', 'protocol_id'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap', 'one_hot', '740-yp_[µm]_one_hot', 'mtg_[µm]_one

### Splitting adata.

In [7]:
train_adata, ood_adatas_dict = split_adata(
    adata,
    config.ood.obs_column,
    config.ood.unique_value_ids,
)
train_adata, ood_adatas_dict.keys()

(AnnData object with n_obs × n_vars = 99517 × 21
     obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type', 'protocol_id'
     var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
     uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap', 'one_hot', '740-yp_[µm]_one_hot', 'mtg_[µm]_

### Applying shared transformations.

In [8]:
train_adata, ood_adata = apply_shared_transformations(
    train_adata,
    ood_adatas_dict,
    standardize_channel_features=config.transforms.standardize_channel_features,
    channel_obsm_key=config.transforms.channel_obsm_key,
    channel_params_uns_key=config.transforms.channel_params_uns_key,
    compute_channel_pcs=config.transforms.compute_channel_pcs,
    pca_obsm_key=config.transforms.pca_obsm_key,
    standardize_scatter_features=config.transforms.standardize_scatter_features,
    scatter_obsm_key=config.transforms.scatter_obsm_key,
    scatter_params_uns_key=config.transforms.scatter_params_uns_key,
)
train_adata, ood_adatas_dict

(AnnData object with n_obs × n_vars = 99517 × 21
     obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type', 'protocol_id'
     var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
     uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap', 'one_hot', '740-yp_[µm]_one_hot', 'mtg_[µm]_

## Model.

### Initializing CFM model.

In [9]:
flow_matching =  FlowMatching(
    flow_type=config.flow_matching.flow_type,
    flow_kwargs=resolve_omegaconf_to_dictionary(config.flow_matching.flow_kwargs),
    coupling_type=config.flow_matching.coupling_type,
    coupling_kwargs=resolve_omegaconf_to_dictionary(config.flow_matching.coupling_kwargs),
    time_sampler=time_samplers.get(config.flow_matching.time_sampler, torch.rand),
    device_id=config.flow_matching.device_id,
    generate_from_noise=config.flow_matching.generate_from_noise,
    noise_distribution=noise_distributions.get(config.flow_matching.noise_distribution, torch.randn),
)

### Setting up data.

In [10]:
flow_matching.prepare_train_data(
    train_adata,
    sample_rep=config.data.sample_rep,
    control_key=config.data.control_key,
    perturbations=config.data.perturbations,
    perturbations_in_obsm=config.data.perturbations_in_obsm,
    perturbation_covariates=config.data.perturbation_covariates,
    perturbation_reps=config.data.perturbation_reps,
    load_target_covariates=config.data.load_target_covariates,
    target_covariates=config.data.target_covariates,
    target_covariates_in_obsm=config.data.target_covariates_in_obsm,
    target_covariates_kwargs=resolve_omegaconf_to_dictionary(config.data.target_covariates_kwargs),
)
ood_data_dict = {
    k: flow_matching.data_manager.get_data(v) for k, v in ood_adatas_dict.items()
}

### Initializing velocity field.

In [11]:
cvf_config = NeuralVelocityFieldConfig(
    flow_matching.train_data.state_data.shape[-1],
    encode_state=config.vf.encode_state,
    state_encoder_output_dim=config.vf.state_encoder_output_dim,
    state_encoder_mlp_kwargs=parse_mlp_config_dictionary(activation_functions, config.vf.state_encoder_mlp_kwargs),
    encode_time=config.vf.encode_time,
    use_sinusoidal_time_features=config.vf.use_sinusoidal_time_features,
    time_features_num_freqs=config.vf.time_features_num_freqs,
    time_features_max_periods=config.vf.time_features_max_periods,
    time_encoder_output_dim=config.vf.time_encoder_output_dim,
    time_encoder_mlp_kwargs=parse_mlp_config_dictionary(activation_functions, config.vf.time_encoder_mlp_kwargs),
    use_guidance=config.vf.use_guidance,
    encode_conditions=config.vf.encode_conditions,
    perturbation_encoder_output_dim=config.vf.perturbation_encoder_output_dim,
    perturbation_layers_before_pooling=parse_nested_mlp_config_dictionary(activation_functions, config.vf.perturbation_layers_before_pooling),
    perturbation_covariates_not_pooled=config.vf.perturbation_covariates_not_pooled,
    perturbation_pooling=config.vf.perturbation_pooling,
    perturbation_pooling_kwargs=config.vf.perturbation_pooling_kwargs,
    perturbation_layers_after_pooling=parse_mlp_config_dictionary(activation_functions, config.vf.perturbation_layers_after_pooling),
    decoder_mlp_kwargs=parse_mlp_config_dictionary(activation_functions, config.vf.decoder_mlp_kwargs),
    use_source_as_condition=config.vf.use_source_as_condition,
    encode_source=config.vf.encode_source, 
    source_encoder_output_dim=config.vf.source_encoder_output_dim,
    source_encoder_mlp_kwargs=parse_mlp_config_dictionary(activation_functions, config.vf.source_encoder_mlp_kwargs),
    conditioning_type=config.vf.conditioning_type,
    n_resnet_blocks=config.vf.n_resnet_blocks,
    resnet_dropout_prob=config.vf.resnet_dropout_prob,
    resnet_normalization=config.vf.resnet_normalization,
    use_classifier_free_guidance=config.vf.use_classifier_free_guidance,
    cfg_null_condition_token=config.vf.cfg_null_condition_token,
)

### Initializing Neural Velocity Field.

In [12]:
flow_matching.prepare_model(
    cvf_config,
    optimizer_class=optimizers.get(config.model.optimizer_id, torch.optim.Adam),
    optimizer_kwargs=resolve_omegaconf_to_dictionary(config.model.optimizer_kwargs),
    lr_scheduler_class=schedulers.get(config.model.scheduler_id, None),
    lr_scheduler_kwargs=resolve_omegaconf_to_dictionary(config.model.lr_scheduler_kwargs),
    lr_scheduler_step=config.model.lr_scheduler_step,
    num_time_steps=config.model.num_time_steps,
    solver_kwargs=resolve_omegaconf_to_dictionary(config.model.solver_kwargs),
)
print(flow_matching.velocity_field)

NeuralVelocityField(
  (vf_modules): ModuleDict(
    (x_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=27, out_features=256, bias=True)
          (1): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Linear(in_features=256, out_features=16, bias=True)
          (1): Identity()
        )
      )
    )
    (time_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=256, out_features=256, bias=True)
          (1): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Linear(in_features=256, out_features=8, bias=True)
          (1): Identity()
        )
      )
    )
    (condition_encoder): ConditionEncoder(
      (modules_dict): ModuleDict(
        (repr_protocol_id_one_hot): MLPBlock(
          (net): Sequential(
            (0): Sequential(
              (0): Linear(in_features=116, out_features=16, bias=True)
              (1): ReLU()
            )
    

### Prepare training callbacks.

In [13]:
wandb_callback = WandBLogger(
    project_name=config.callbacks.wandb_project_name,
    log_dir=config.paths.log_dir,
    config=config,
    **resolve_omegaconf_to_dictionary(config.callbacks.wandb_kwargs),
)
metrics_callback = MetricsCallBack(
    config.callbacks.metrics,
    state_transforms=state_transforms.get(config.training.state_transforms, None),
)
callbacks = TrainingCallBacks(
    [
        wandb_callback,
        metrics_callback
    ]
)

### Train CFM.

In [ ]:
flow_matching.train(
    num_training_steps=config.training.num_training_steps,
    valid_freq=config.training.valid_freq,
    train_batch_size=config.training.train_batch_size,
    validation_batch_size=config.training.validation_batch_size,
    state_transforms=state_transforms.get(config.training.state_transforms, None),
    callbacks=callbacks,
    grad_steps_log_interval=config.training.grad_steps_log_interval,
    num_treatments_to_load=config.training.num_treatments_to_load,
    num_samples_per_validation_step=config.training.num_samples_per_validation_step,
    cfg_prob_unconditional=config.training.cfg_prob_unconditional,
    validation_cfg_guidance_strength=config.training.validation_cfg_guidance_strength,
    num_grad_accumulation_steps=config.training.num_grad_accumulation_steps,
    close_wandb_connection=False,
)

  0%|          | 0/100 [00:00<?, ?it/s]

### Validation.

In [ ]:
N = 1000
num_time_steps = 100
sep = "+"
splits_metrics_dict = {}
for split_id, split_data in ood_data_dict.items():
    splits_metrics_dict[split_id] = validate_on_ood_data(
        N,
        flow_matching,
        split_data,
        callbacks,
        sep=sep
    )
callbacks.run_on_train_end()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


5_100_10_10_2.5_750_1120.0_0.0_50_0_0_0_5.0_20_21_energy_distance,▁
5_100_10_10_2.5_750_1120.0_0.0_50_0_0_0_5.0_20_21_maximum_mean_discrepancy,▁
5_100_10_10_2.5_750_1120.0_0.0_50_0_0_0_5.0_20_21_r_squared,▁
5_100_10_10_2.5_750_1120.0_0.0_50_0_0_0_5.0_20_21_sinkhorn_divergence,▁
5_100_10_10_2.5_750_1120.0_0.0_50_0_0_0_5.0_20_21_wasserstein_distance,▁
condition_value_energy_distance,▁
condition_value_maximum_mean_discrepancy,▁
condition_value_r_squared,▁
condition_value_sinkhorn_divergence,▁
condition_value_wasserstein_distance,▁
5_100_10_10_2.5_750_1120.0_0.0_50_0_0_0_5.0_20_21_energy_distance,702985873518.5916


/home/icb/lorenzo.consoli/miniconda3/envs/sc_exp_design/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:631: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/home/icb/lorenzo.consoli/miniconda3/envs/sc_exp_design/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:643: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)
/home/icb/lorenzo.consoli/miniconda3/envs/sc_exp_design/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:631: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/home/icb/lorenzo.consoli/miniconda3/envs/sc_exp_design/lib/python3.12/site-packages/ot/bregman/_sinkhorn.py:643: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)
